# Non-linearity probes

How much does non-linearity add to predictability? Two probes per walk-forward split
(train days i-TRAIN_DAYS..i-1, test day i), using the tuned XGB params from `xgboost.ipynb`:

1. Hybrid: OLS plus an XGBoost booster on the OLS train residuals (frozen params incl. `n_estimators`, no early stopping).
2. Linearity ladder: ridge on cumulative feature expansions, `linear` → `asym` → `curv` → `inter`,
   ridge penalty scaled with the column count (D015).

All legs go into one `daily_diagnostics.parquet` with a `model_leg` column.
OLS / XGB baselines are not refit here; the manifest records the runs to compare against
(see `model_evaluation.ipynb`, section 4).


In [ ]:
import os, warnings, sys
import json
import time
from itertools import combinations
from datetime import datetime, timezone
from tqdm.auto import tqdm

import numpy as np
import pandas as pd

from sklearn.linear_model import LinearRegression, Ridge
from xgboost import XGBRegressor

sys.path.append(os.path.dirname(os.getcwd()))
import importlib
import utils.data_processing as du
import utils.execution as execution
import utils.pipeline as pipeline
import utils.workers as workers

importlib.reload(du)
importlib.reload(execution)
importlib.reload(pipeline)
importlib.reload(xw)

In [ ]:
# Config
warnings.filterwarnings("ignore")

RUN_SYMBOLS = du.SYMBOLS[:1]

LOCAL = True   # local subset: 2 stocks, 20 days (one missing), 3-day tuning block, 1 pair
if LOCAL:
    RUN_SYMBOLS = du.SYMBOLS[:2]

TRAIN_DAYS = 1                # training window in days
SEED = 0                      # tuning seed
FEATURES = ["L1-QImb", "MicroPrice"]   # feature names; all lags of each are used
# same NAME as xgboost.ipynb
NAME = f"{TRAIN_DAYS}-days_{SEED}-seed_{'standard' if set(FEATURES) == {'L1-QImb', 'MicroPrice'} else 'expanded'}-features"
HORIZONS = ["100ms", "2s", "30s", "5m"]


PURPOSE = ""  # free text: why this run exists

# baseline runs for the evaluation notebook
REF_REGRESSION_RUN = "1-days_standard-features"
REF_XGB_RUN = None            # set once a matching XGBoost run exists

# Ladder settings
CLIP_SD = 5.0                 # winsorization of standardized features (see utils)
# ridge alpha scaled with column count (D015)
RIDGE_ALPHA = 1.0             # anchor alpha_0 at the linear rung

# fixed XGB settings
XGB_PARAMS = dict(
    tree_method="hist",
    max_bin=128,
    device=execution.select_device(),   # least-used GPU, else "cpu"
    n_jobs=-1,
    random_state=0,
)

PARENT = os.path.dirname(os.getcwd())
DATA_ROOT = f"{PARENT}/data/processed"
OUTPUT_ROOT = f"{PARENT}/model_outputs/Nonlinearity"
XGB_RUN_DIR = f"{PARENT}/model_outputs/XGBoost/runs/{NAME}"   # tuned params live in its manifest

In [ ]:
# Setup
# frozen tuned params
if not os.path.exists(f"{XGB_RUN_DIR}/manifest.json"):
    raise FileNotFoundError(f"{XGB_RUN_DIR} not found - run xgboost.ipynb Part 1 with NAME = {NAME!r} first.")
with open(f"{XGB_RUN_DIR}/manifest.json") as f:
    _xgb = json.load(f)
TUNED_PARAMS = _xgb["params"]
if not TUNED_PARAMS:
    raise ValueError(f"{XGB_RUN_DIR}: winners not frozen yet")
TUNE_DATES = _xgb["tune_dates"]

# walk-forward starts after the tuning block
# empty on the local subset
SD = [d for d in du.SAMPLE_DATES if d > max(TUNE_DATES)]

FEATURE_COLS, TARGET_COLS = workers.feature_target_cols(RUN_SYMBOLS[0], HORIZONS, FEATURES)

MODEL_LEGS = ["hybrid", "ladder_linear", "ladder_asym", "ladder_curv", "ladder_inter"]

RUN_DIR = pipeline.start_run(OUTPUT_ROOT, NAME, {
    "feature_set_id": "+".join(FEATURES),
    "symbols": RUN_SYMBOLS,
    "horizons": HORIZONS,
    "feature_cols": FEATURE_COLS,
    "target_cols": TARGET_COLS,
    "purpose": PURPOSE,
    "hp_config": {
        "ref_regression_run": REF_REGRESSION_RUN,
        "ref_xgboost_run": REF_XGB_RUN,
        "tuning": {
            "source": XGB_RUN_DIR,
            "tune_dates": TUNE_DATES,
            "train_days": TRAIN_DAYS,
            "fixed_params": {k: v for k, v in XGB_PARAMS.items() if k != "device"},
            "params": {s: TUNED_PARAMS[s] for s in RUN_SYMBOLS},
        },
        "ladder": {"clip_sd": CLIP_SD, "ridge_alpha": RIDGE_ALPHA,
                   "alpha_rule": "alpha = ridge_alpha * n_cols / n_features"},
        "model_legs": MODEL_LEGS,
    },
})


daily_results = []

In [ ]:
# Linearity ladder: standardized features and design matrices per rung
def standardize_features(
        X_train,
        X_test,
        clip_sd: float = 5.0
) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """Z-score with train mean/std and clip at +-clip_sd. Returns (Z_train, Z_test, mean, sd)."""
    mean = X_train.mean(axis=0)
    sd = X_train.std(axis=0)
    safe_sd = np.where(sd > 0, sd, 1.0)

    Z_train = np.clip((X_train - mean) / safe_sd, -clip_sd, clip_sd)
    Z_test = np.clip((X_test - mean) / safe_sd, -clip_sd, clip_sd)

    zero_var = sd == 0
    if zero_var.any():
        Z_train[:, zero_var] = 0.0
        Z_test[:, zero_var] = 0.0

    return (Z_train.astype(np.float32), Z_test.astype(np.float32), mean, sd)


def ladder_blocks(
        Z_train,
        Z_test,
) -> dict[str, tuple[np.ndarray, np.ndarray]]:
    """Design matrices for the linearity ladder, {rung: (A_train, A_test)}.

        linear: Z
        asym:   + max(Z, 0)
        curv:   + Z*|Z|, Z**2
        inter:  + pairwise products
    """
    def stack(blocks):
        return tuple(
            np.column_stack(mats).astype(np.float32)
            for mats in zip(*blocks)
        )

    blocks = [(Z_train, Z_test)]
    out = {"linear": stack(blocks)}

    blocks.append((np.maximum(Z_train, 0), np.maximum(Z_test, 0)))
    out["asym"] = stack(blocks)

    blocks.append((Z_train * np.abs(Z_train), Z_test * np.abs(Z_test)))
    blocks.append((Z_train ** 2, Z_test ** 2))
    out["curv"] = stack(blocks)

    pairs = list(combinations(range(Z_train.shape[1]), 2))
    blocks.append((
        np.column_stack([Z_train[:, a] * Z_train[:, b] for a, b in pairs]),
        np.column_stack([Z_test[:, a] * Z_test[:, b] for a, b in pairs]),
    ))
    out["inter"] = stack(blocks)

    return out

In [ ]:
# Walk-forward
GLOBAL_START = time.perf_counter()

for symbol in tqdm(RUN_SYMBOLS, desc="Processing symbols", unit="symbol"):
    day_cache = pipeline.load_day_cache(DATA_ROOT, symbol, SD, FEATURE_COLS, TARGET_COLS)
    sd_sym = sorted(day_cache)  # days actually available for this symbol
    symbol_params = TUNED_PARAMS[symbol]   # {target: frozen params incl. n_estimators}

    for i in tqdm(range(TRAIN_DAYS, len(sd_sym)), desc=f"{symbol}", leave=False, unit="split"):
        train_days, test_day = sd_sym[i - TRAIN_DAYS:i], sd_sym[i]
        X_train, Y_train = workers.stack_days(day_cache, train_days)
        test_cache = day_cache[test_day]
        X_test, Y_test = test_cache["X"], test_cache["Y"]

        # OLS in scaled target units
        ols = LinearRegression().fit(X_train, Y_train * pipeline.TARGET_SCALE)
        ols_resid_train_scaled = (Y_train * pipeline.TARGET_SCALE - ols.predict(X_train)).astype(np.float32)
        ols_pred_test = (ols.predict(X_test) / pipeline.TARGET_SCALE).astype(np.float32)

        # ladder inputs
        Z_train, Z_test, _, _ = standardize_features(X_train, X_test, clip_sd=CLIP_SD)
        blocks = ladder_blocks(Z_train, Z_test)

        leg_resid = {leg: np.empty_like(Y_test) for leg in MODEL_LEGS}

        for j, target in enumerate(TARGET_COLS):
            # hybrid: booster on OLS train residuals
            booster = XGBRegressor(**{**XGB_PARAMS, **symbol_params[target]})
            booster.fit(X_train, ols_resid_train_scaled[:, j])
            hybrid_pred = ols_pred_test[:, j] + booster.predict(X_test) / pipeline.TARGET_SCALE
            leg_resid["hybrid"][:, j] = Y_test[:, j] - hybrid_pred

            # ladder
            for rung, (A_train, A_test) in blocks.items():
                alpha = RIDGE_ALPHA * A_train.shape[1] / Z_train.shape[1]
                model = Ridge(alpha=alpha).fit(A_train, Y_train[:, j] * pipeline.TARGET_SCALE)
                leg_resid[f"ladder_{rung}"][:, j] = Y_test[:, j] - model.predict(A_test) / pipeline.TARGET_SCALE


        for leg in MODEL_LEGS:
            rows = pipeline.daily_diagnostic_rows(
                resid=leg_resid[leg],
                Y_test=Y_test,
                target_cols=TARGET_COLS,
                train_day=train_days[-1],   # last day of the window (length in the manifest)
                test_day=test_day,
                symbol=symbol,
                run_id=NAME,
                n_train=X_train.shape[0],
                n_test=X_test.shape[0],
            )
            daily_results += [dict(r, model_leg=leg) for r in rows]

tqdm.write(f"\nTOTAL PIPELINE TIME: {time.perf_counter()-GLOBAL_START:.2f}s")

# Save
pd.DataFrame(daily_results).to_parquet(f"{RUN_DIR}/daily_diagnostics.parquet", index=False)
with open(f"{RUN_DIR}/manifest.json") as f:
    manifest = json.load(f)
manifest["status"] = "complete"
manifest["runtime_seconds"] = round((datetime.now(timezone.utc) - datetime.fromisoformat(manifest["created_at"])).total_seconds(), 1)
with open(f"{RUN_DIR}/manifest.json", "w") as f:
    json.dump(manifest, f, indent=2, default=str)

In [ ]:
daily_out = pd.read_parquet(f"{RUN_DIR}/daily_diagnostics.parquet")
display(daily_out.groupby("model_leg")["mse_ratio"].describe().round(4))